# D2 Robustness Check: 5-Fold Cross-Validation

Model_Comparison_D2 picked model configurations using one validation split and reported final numbers on one test split. This notebook checks whether the relative ranking of those three configurations holds up if the training data is split differently, using 5-fold cross-validation on the training portion only.

This is a robustness check, not a new model-selection or tuning exercise. No hyperparameters are changed here.

## Why only the training portion

The validation and test sets were already used in Model_Comparison_D2 (validation for model/threshold selection, test for the final locked evaluation). Reusing either one here would leak information into a check that is supposed to be independent of that process. So this notebook only uses the original 42,394-row training portion, split again internally into 5 folds.

In [1]:
import pandas as pd
import numpy as np

pumf = pd.read_csv("../Data_Données/pumf_cchs.csv")

print("PUMF shape:", pumf.shape)

PUMF shape: (67079, 255)


In [2]:
model_data = pumf[pumf["CCC_05"].isin([1, 2])].copy()

model_data["target"] = (model_data["CCC_05"] == 1).astype(int)

print("Modelling population:", model_data.shape[0])
print(model_data["target"].value_counts().sort_index())

Modelling population: 66242
target
0    60248
1     5994
Name: count, dtype: int64


## Feature set

Same `D2_Core_Hypertension_Cholesterol` feature set as Experiment D2 and Model_Comparison_D2: the 7 C features plus `CCC_80` plus `CCC_90`.

In [3]:
FEATURES = [
    "DHHGAGE",
    "DHH_SEX",
    "EDDVH3",
    "BMI_CLASS",
    "INCDGHH",
    "SDCDGIMM",
    "GEOGPRV",
    "CCC_80",
    "CCC_90"
]

print("Number of features:", len(FEATURES))
print(FEATURES)

Number of features: 9
['DHHGAGE', 'DHH_SEX', 'EDDVH3', 'BMI_CLASS', 'INCDGHH', 'SDCDGIMM', 'GEOGPRV', 'CCC_80', 'CCC_90']


In [4]:
# Same BMI harmonization as C, D1, D2
model_data["BMI_CLASS"] = np.nan

youth_mask = model_data["DHHGAGE"] == 1
adult_mask = model_data["DHHGAGE"].isin([2, 3, 4, 5])

model_data.loc[youth_mask, "BMI_CLASS"] = model_data.loc[youth_mask, "HWTDGWHO"]
model_data.loc[adult_mask, "BMI_CLASS"] = model_data.loc[adult_mask, "HWTDGISW"]

print(model_data["BMI_CLASS"].value_counts(dropna=False).sort_index())

BMI_CLASS
1.0    26053
2.0    37045
6.0       32
9.0     3112
Name: count, dtype: int64


In [5]:
SPECIAL_CODES = {
    "DHHGAGE": [],
    "DHH_SEX": [],
    "EDDVH3": [9],
    "BMI_CLASS": [6, 9],
    "INCDGHH": [9],
    "SDCDGIMM": [9],
    "GEOGPRV": [],
    "CCC_80": [9],
    "CCC_90": [9]
}

def apply_special_codes(df, special_codes):
    result = df.copy()

    for column, codes in special_codes.items():
        if column in result.columns:
            result[column] = result[column].replace(codes, np.nan)

    return result

clean_model_data = apply_special_codes(model_data, SPECIAL_CODES)

print("Missing values after special-code handling:")
print(clean_model_data[FEATURES].isna().sum())

Missing values after special-code handling:
DHHGAGE         0
DHH_SEX         0
EDDVH3       2276
BMI_CLASS    3144
INCDGHH       947
SDCDGIMM      835
GEOGPRV         0
CCC_80        494
CCC_90        155
dtype: int64


## Recreate the original split, keep only the training portion

Same population, seed, and stratified split as every other D2 notebook. Only `train_idx` (42,394 rows) is used below. Validation and test indices are recreated for completeness but not touched.

In [6]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

train_val_idx, test_idx = train_test_split(
    model_data.index,
    test_size=0.20,
    stratify=model_data["target"],
    random_state=RANDOM_STATE
)

train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.20,
    stratify=model_data.loc[train_val_idx, "target"],
    random_state=RANDOM_STATE
)

print("Training portion used for CV:", len(train_idx))
print("Validation (not used here):", len(val_idx))
print("Test (not used here):", len(test_idx))

Training portion used for CV: 42394
Validation (not used here): 10599
Test (not used here): 13249


In [7]:
X_train_cv = clean_model_data.loc[train_idx, FEATURES].reset_index(drop=True)
y_train_cv = model_data.loc[train_idx, "target"].reset_index(drop=True)

print(X_train_cv.shape, y_train_cv.shape)
print(y_train_cv.value_counts())

(42394, 9) (42394,)
target
0    38558
1     3836
Name: count, dtype: int64


## 5-fold setup

`StratifiedKFold` with 5 folds, `shuffle=True`, `random_state=42`, applied only to the training portion above. Preprocessing (imputation + one-hot encoding) is fitted separately inside each fold by putting it in the same `Pipeline` as the model, so no fold sees information from outside its own training split.

In [8]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (fold_train_idx, fold_val_idx) in enumerate(cv.split(X_train_cv, y_train_cv), start=1):
    print(f"Fold {fold}: train={len(fold_train_idx)}, held-out={len(fold_val_idx)}")

Fold 1: train=33915, held-out=8479
Fold 2: train=33915, held-out=8479
Fold 3: train=33915, held-out=8479
Fold 4: train=33915, held-out=8479
Fold 5: train=33916, held-out=8478


## Fixed model configurations

Same three configurations already selected in Model_Comparison_D2. Nothing is retuned here.

- Logistic Regression: `class_weight="balanced"`, `max_iter=1000`, `random_state=42`
- Random Forest: `n_estimators=200`, `max_depth=10`, `class_weight="balanced"`, `random_state=42`
- Gradient Boosting: `n_estimators=200`, `learning_rate=0.05`, `max_depth=3`, `random_state=42`, with `sample_weight` from `compute_sample_weight("balanced", ...)` computed on each fold's own training rows (same imbalance handling as before, just recomputed per fold instead of once).

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

def build_pipeline(classifier):
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer([
        ("categorical", categorical_pipeline, FEATURES)
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

pipelines = {
    "Logistic Regression": build_pipeline(
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
    ),
    "Random Forest": build_pipeline(
        RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1)
    ),
    "Gradient Boosting": build_pipeline(
        GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
    )
}

print("Pipelines built for:", list(pipelines.keys()))

Pipelines built for: ['Logistic Regression', 'Random Forest', 'Gradient Boosting']


## Run 5-fold CV for each model

Main metrics are ROC-AUC, PR-AUC (average precision), and log loss. F1 is not used here because it depends on a classification threshold, and no threshold selection is part of this check.

In [10]:
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
from sklearn.utils.class_weight import compute_sample_weight

fold_results = []

for model_name, pipeline in pipelines.items():
    for fold, (fold_train_idx, fold_val_idx) in enumerate(cv.split(X_train_cv, y_train_cv), start=1):
        X_fold_train = X_train_cv.iloc[fold_train_idx]
        y_fold_train = y_train_cv.iloc[fold_train_idx]
        X_fold_val = X_train_cv.iloc[fold_val_idx]
        y_fold_val = y_train_cv.iloc[fold_val_idx]

        if model_name == "Gradient Boosting":
            fold_sample_weight = compute_sample_weight("balanced", y_fold_train)
            pipeline.fit(X_fold_train, y_fold_train, classifier__sample_weight=fold_sample_weight)
        else:
            pipeline.fit(X_fold_train, y_fold_train)

        fold_prob = pipeline.predict_proba(X_fold_val)[:, 1]

        fold_results.append({
            "model": model_name,
            "fold": fold,
            "roc_auc": roc_auc_score(y_fold_val, fold_prob),
            "pr_auc": average_precision_score(y_fold_val, fold_prob),
            "log_loss": log_loss(y_fold_val, fold_prob)
        })

fold_results = pd.DataFrame(fold_results)
print(fold_results.round(4))

                  model  fold  roc_auc  pr_auc  log_loss
0   Logistic Regression     1   0.8281  0.2823    0.5282
1   Logistic Regression     2   0.8019  0.2502    0.5351
2   Logistic Regression     3   0.8208  0.2808    0.5353
3   Logistic Regression     4   0.8152  0.2688    0.5325
4   Logistic Regression     5   0.8272  0.3032    0.5248
5         Random Forest     1   0.8270  0.2799    0.5014
6         Random Forest     2   0.8029  0.2582    0.5066
7         Random Forest     3   0.8192  0.2793    0.5074
8         Random Forest     4   0.8159  0.2735    0.5032
9         Random Forest     5   0.8238  0.2814    0.5003
10    Gradient Boosting     1   0.8304  0.2882    0.5198
11    Gradient Boosting     2   0.8057  0.2566    0.5251
12    Gradient Boosting     3   0.8207  0.2865    0.5263
13    Gradient Boosting     4   0.8167  0.2746    0.5233
14    Gradient Boosting     5   0.8281  0.2981    0.5178


## Fold-level results

In [11]:
print(fold_results.pivot(index="fold", columns="model", values="roc_auc").round(4))
print()
print(fold_results.pivot(index="fold", columns="model", values="pr_auc").round(4))
print()
print(fold_results.pivot(index="fold", columns="model", values="log_loss").round(4))

model  Gradient Boosting  Logistic Regression  Random Forest
fold                                                        
1                 0.8304               0.8281         0.8270
2                 0.8057               0.8019         0.8029
3                 0.8207               0.8208         0.8192
4                 0.8167               0.8152         0.8159
5                 0.8281               0.8272         0.8238

model  Gradient Boosting  Logistic Regression  Random Forest
fold                                                        
1                 0.2882               0.2823         0.2799
2                 0.2566               0.2502         0.2582
3                 0.2865               0.2808         0.2793
4                 0.2746               0.2688         0.2735
5                 0.2981               0.3032         0.2814

model  Gradient Boosting  Logistic Regression  Random Forest
fold                                                        
1                 0.51

## Mean ± standard deviation across folds

In [12]:
summary = fold_results.groupby("model")[["roc_auc", "pr_auc", "log_loss"]].agg(["mean", "std"])
print(summary.round(4))

                    roc_auc          pr_auc         log_loss        
                       mean     std    mean     std     mean     std
model                                                               
Gradient Boosting    0.8203  0.0099  0.2808  0.0159   0.5225  0.0036
Logistic Regression  0.8186  0.0107  0.2770  0.0195   0.5312  0.0046
Random Forest        0.8177  0.0093  0.2744  0.0096   0.5038  0.0031


## Save results

In [13]:
summary_flat = fold_results.groupby("model")[["roc_auc", "pr_auc", "log_loss"]].agg(["mean", "std"]).reset_index()
summary_flat.columns = ["model", "roc_auc_mean", "roc_auc_std", "pr_auc_mean", "pr_auc_std", "log_loss_mean", "log_loss_std"]

fold_results_export = fold_results.copy()
fold_results_export["fold"] = fold_results_export["fold"].astype(str)

summary_rows_mean = summary_flat[["model", "roc_auc_mean", "pr_auc_mean", "log_loss_mean"]].rename(
    columns={"roc_auc_mean": "roc_auc", "pr_auc_mean": "pr_auc", "log_loss_mean": "log_loss"}
)
summary_rows_mean["fold"] = "mean"

summary_rows_std = summary_flat[["model", "roc_auc_std", "pr_auc_std", "log_loss_std"]].rename(
    columns={"roc_auc_std": "roc_auc", "pr_auc_std": "pr_auc", "log_loss_std": "log_loss"}
)
summary_rows_std["fold"] = "std"

full_results = pd.concat([fold_results_export, summary_rows_mean, summary_rows_std], ignore_index=True)
full_results = full_results[["model", "fold", "roc_auc", "pr_auc", "log_loss"]]

full_results.to_csv("d2_robustness_5fold_results.csv", index=False)

print("Saved: d2_robustness_5fold_results.csv")
full_results

Saved: d2_robustness_5fold_results.csv


,model,fold,roc_auc,pr_auc,log_loss
0,Logistic Regression,1,0.828106,0.282286,0.528208
1,Logistic Regression,2,0.801938,0.250160,0.535058
2,Logistic Regression,3,0.820798,0.280815,0.535259
3,Logistic Regression,4,0.815172,0.268756,0.532479
4,Logistic Regression,5,0.827181,0.303191,0.524803
5,Random Forest,1,0.826966,0.279898,0.501437
6,Random Forest,2,0.802903,0.258195,0.506625
7,Random Forest,3,0.819212,0.279253,0.507423
8,Random Forest,4,0.815893,0.273462,0.503215
9,Random Forest,5,0.823775,0.281369,0.500300


## Interpretation

| Model | ROC-AUC (mean ± std) | PR-AUC (mean ± std) | Log loss (mean ± std) |
|---|---|---|---|
| Logistic Regression | 0.8186 ± 0.0107 | 0.2770 ± 0.0195 | 0.5312 ± 0.0046 |
| Random Forest | 0.8177 ± 0.0093 | 0.2744 ± 0.0096 | 0.5038 ± 0.0031 |
| Gradient Boosting | 0.8203 ± 0.0099 | 0.2808 ± 0.0159 | 0.5225 ± 0.0036 |

The gaps between the three models' means are small (about 0.001-0.003 on ROC-AUC, 0.003-0.006 on PR-AUC) and are close to the size of their own fold-to-fold standard deviations, so the models overlap substantially across folds. This matches what Model_Comparison_D2 already found on the single test split: no model clearly dominates.

Two of the three CV rankings agree with the original test-set ranking. On ROC-AUC, Gradient Boosting is highest, Logistic Regression second, Random Forest last in both the 5-fold CV and the test results. On log loss, Random Forest is lowest (best), then Gradient Boosting, then Logistic Regression, again in both the CV and the test results. On PR-AUC the ranking does not fully agree: the CV mean has Gradient Boosting slightly ahead of Logistic Regression, while the test set had Logistic Regression slightly ahead of Gradient Boosting. The gap in both cases is small (well under 0.01), so this looks like the two models trading a very close second/third place rather than a real disagreement about which model is better.

Overall, the ranking of these three model configurations is reasonably stable across different training folds, especially on ROC-AUC and log loss. This is consistent with the earlier conclusion that model class matters much less than the feature representation for the D2 feature set.

## Limitation

This 5-fold CV only measures how much ROC-AUC, PR-AUC, and log loss move around when the training portion is split differently. It does not say anything about whether the single held-out test split used in Model_Comparison_D2 was itself representative of the population, and it does not replace or override the locked test results reported there. It is a check on training-side variability only.